#  Task 1 — HadithBot
**Pipeline:** Clone Dataset → Preprocess → MiniLM Embed → FAISS Store → Similarity Search

In [ ]:
# ── STEP 1: Clone the LK Hadith Corpus from GitHub ──────────────────
!git clone https://github.com/ShathaTm/LK-Hadith-Corpus.git

In [ ]:
# ── STEP 2: Load and Preprocess Hadith CSV Files ────────────────────
import glob, re, pandas as pd, numpy as np

path = '/content/LK-Hadith-Corpus'
files = sorted(glob.glob(path + '//**/*.csv', recursive=True))
print(f'Found {len(files)} CSV files')

columns = [
    'Chapter_Number', 'Chapter_English', 'Chapter_Arabic',
    'Section_Number', 'Section_English', 'Section_Arabic',
    'Hadith_Number',
    'English_Hadith', 'English_Isnad', 'English_Matn', 'English_Grade',
    'Arabic_Hadith', 'Arabic_Isnad', 'Arabic_Matn', 'Arabic_Grade'
]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)              # remove extra spaces
    return text.strip()

all_hadith = []
for file in files:
    df = pd.read_csv(file, names=columns, skiprows=1)
    df['Cleaned_Hadith'] = df['English_Hadith'].astype(str).apply(clean_text)
    all_hadith.extend(df[columns + ['Cleaned_Hadith']].values.tolist())

hadith_df = pd.DataFrame(all_hadith, columns=columns + ['Cleaned_Hadith'])
hadith_df.to_csv('cleaned_hadith_data.csv', index=False)
print(f'Total Hadiths Loaded: {len(hadith_df)}')
hadith_df.head(2)

In [ ]:
# ── STEP 3: Install Libraries & Load MiniLM Model ───────────────────
!pip install sentence-transformers faiss-cpu -q

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Model loaded!')

In [ ]:
# ── STEP 4: Generate Embeddings ──────────────────────────────────────
embeddings = model.encode(hadith_df['Cleaned_Hadith'].values, show_progress_bar=True)
embeddings = np.array(embeddings)
np.save('hadith_embeddings.npy', embeddings)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── STEP 5: Build FAISS Index ─────────────────────────────────────────
import faiss

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dim)   # L2 = Euclidean distance
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'faiss_index.index')
print(f'FAISS index built with {faiss_index.ntotal} vectors')

In [ ]:
# ── STEP 6: Search Function ───────────────────────────────────────────
def get_similar_hadith(query, count=5):
    query_embedding = model.encode([clean_text(query)])
    distances, indices = faiss_index.search(query_embedding, count)
    print(f'\n🔍 Query: "{query}"\n' + '='*60)
    for i in range(count):
        print(f'\nHadith #{i+1}  [Distance: {distances[0][i]:.4f}]')
        print(hadith_df['English_Hadith'].iloc[indices[0][i]])
        print('-'*60)

# ── DEMO ──────────────────────────────────────────────────────────────
get_similar_hadith('How many prayers are there?')
get_similar_hadith('What is fasting?')

In [ ]:
# ── STEP 1: Create Q&A Dataset ───────────────────────────────────────
import pandas as pd

qa_data = [
    {'question': 'What are the admission requirements for undergraduate programs?',
     'answer': 'Undergraduate admission requires minimum 60% marks in FSc/A-Levels. Submit transcripts, CNIC, two passport photos, and complete the online application form.'},
    {'question': 'What is the last date to apply for fall admission?',
     'answer': 'Fall semester deadline is August 15. Spring semester closes January 10. Check the official website for current year deadlines.'},
    {'question': 'How do I apply online for admission?',
     'answer': 'Visit the official website, click Apply Now, create an account with your CNIC, fill in details, upload documents, and pay the application fee online.'},
    {'question': 'What is the application fee?',
     'answer': 'Application fee is PKR 2,000 for undergraduate and PKR 2,500 for postgraduate programs. Non-refundable.'},
    {'question': 'What documents are required for admission?',
     'answer': 'Required: academic certificates (original + attested), CNIC/B-Form, domicile certificate, 2 passport photos, character certificate, and fee receipt.'},
    {'question': 'Is there an entry test for admission?',
     'answer': 'Yes. Students must pass the university entry test or submit SAT/GAT/NTS scores. Test covers Math, English, and subject topics. Minimum 50% required.'},
    {'question': 'What is the merit calculation formula?',
     'answer': 'Merit = FSc/A-Level marks (50%) + Entry Test score (40%) + Hafiz-e-Quran bonus (10%). Results shown on student portal.'},
    {'question': 'What scholarships are available?',
     'answer': 'Need-based (up to 100% fee waiver), merit-based (top 3 per department), HEC need-based, and PEEF scholarships are available.'},
    {'question': 'How do I apply for hostel?',
     'answer': 'Apply via Hostel Office or online portal. Priority for outstation students. Fee: PKR 15,000–25,000 per semester including meals.'},
    {'question': 'What programs does the university offer?',
     'answer': 'BS programs in CS, SE, BBA, EE, CE and more. MS and PhD available in most departments.'},
    {'question': 'What is the duration of the BS program?',
     'answer': 'BS is 4 years (8 semesters). Each semester is 18 weeks. Students must complete 130–136 credit hours.'},
    {'question': 'Can I transfer from another university?',
     'answer': 'Yes, from HEC-recognized universities. Max 50% credit hours transferable. Minimum CGPA 2.5. Apply within the first two weeks of semester.'},
    {'question': 'What is the fee structure for BS Computer Science?',
     'answer': 'PKR 3,500 per credit hour. Typical semester (18 credits) = PKR 63,000 + ~PKR 5,000 lab/library/activity fees.'},
    {'question': 'Is there a fee installment plan?',
     'answer': 'Yes. Pay 50% at admission, remaining 50% within six weeks. PKR 500 processing fee applies.'},
    {'question': 'How do I check my admission status?',
     'answer': 'Login to portal.university.edu.pk using your CNIC and registration password.'},
    {'question': 'What is the grading system?',
     'answer': '4.0 GPA scale. A=4.0, B=3.0, C=2.0, D=1.0, F=0.0. Minimum CGPA 2.0 required to stay enrolled.'},
    {'question': 'Is there an age limit for admission?',
     'answer': 'Undergraduate: max 22 years at admission. No age limit for postgraduate. Up to 2 years relaxation in special cases.'},
    {'question': 'Can international students apply?',
     'answer': 'Yes. Provide HEC equivalency certificate, valid passport, student visa, and NOC from embassy.'},
    {'question': 'What are the contact details of the admission office?',
     'answer': 'Phone: 042-XXXXXXXX | Email: admissions@university.edu.pk | Hours: 9 AM – 5 PM, Mon–Sat'},
    {'question': 'What happens if I miss the admission deadline?',
     'answer': 'Late applications may be considered if seats remain. Late fee of PKR 1,000 applies. Contact Admissions Office immediately.'},
    {'question': 'What is the minimum CGPA to maintain enrollment?',
     'answer': 'Minimum CGPA is 2.0. Below this triggers academic probation for one semester. Non-improvement results in suspension.'},
    {'question': 'Are there reserved seats for special categories?',
     'answer': 'Yes: Open merit (50%), Self-finance (25%), University employees children (5%), Foreign nationals (2%), Sports (2%), Disabled (2%).'},
    {'question': 'How can I get my admission letter?',
     'answer': 'Download from student portal after admission is confirmed. Physical copy available at Admissions Office.'},
    {'question': 'What is the process for getting a student ID card?',
     'answer': 'After fee payment, visit IT department with admission form and one passport photo. ID ready in 3–5 working days.'},
    {'question': 'Can I defer my admission to the next semester?',
     'answer': 'Yes, deferral allowed for one semester with valid reason. Submit request within two weeks of admission. Subject to approval.'},
]

df = pd.DataFrame(qa_data)
df['cleaned_question'] = df['question'].str.lower().str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.strip()
df.to_csv('admission_qna.csv', index=False)
print(f'Dataset saved: {len(df)} Q&A pairs')
df.head(3)

In [ ]:
# ── STEP 2: Generate MiniLM Embeddings for Questions ─────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = np.array(model.encode(df['cleaned_question'].values, show_progress_bar=True))
np.save('admission_embeddings.npy', embeddings)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── STEP 3: Build FAISS Index ─────────────────────────────────────────
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, 'admission_faiss.index')
print(f'FAISS index ready: {index.ntotal} vectors stored')

In [ ]:
# ── STEP 4: Search Function ───────────────────────────────────────────
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def ask_chatbot(query, top_k=3):
    q_emb = model.encode([clean_text(query)])
    distances, indices = index.search(np.array(q_emb), top_k)
    print(f'\n🎓 Query: "{query}"\n' + '='*65)
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        print(f'\nResult #{i+1} [Distance: {dist:.4f}]')
        print(f'Q: {df["question"].iloc[idx]}')
        print(f'A: {df["answer"].iloc[idx]}')
        print('-'*65)

# ── TEST ──────────────────────────────────────────────────────────────
ask_chatbot('How do I get admission?')
ask_chatbot('Tell me about scholarships')
ask_chatbot('What is the fee for computer science?')

In [ ]:
# ── STEP 5: Launch Flask Web App ──────────────────────────────────────
# Run this in a terminal (not Colab) after setting up the project:
#   python app.py
# Then open: http://127.0.0.1:5000

# ── OR run ngrok in Colab for public URL ─────────────────────────────
# !pip install flask pyngrok -q
# from pyngrok import ngrok
# import threading
# threading.Thread(target=lambda: app.run(port=5000)).start()
# public_url = ngrok.connect(5000)
# print('Public URL:', public_url)
print('Flask app ready! Run: python app.py')